# 01 – Feasibility & Ticker Validation

**Project:** Technology & Fintech Market Intelligence and Regime Analyzer  
**Phase:** 1 – Feasibility and scope  
**Goal:** Validate the candidate universe, yfinance data availability, local MongoDB connectivity, and the text source before any modelling work begins.

In [2]:
from __future__ import annotations

import sys
from pathlib import Path
from datetime import datetime, timedelta
from typing import Any, Dict, List

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import settings, TICKERS, CATEGORIES
from config.tickers import get_primary_category
from src.utils import setup_logging, get_logger, utc_now
from src.db.connection import get_client, get_db, close_client
from src.db.collections import ensure_indexes
from src.data_pipeline.yfinance_client import fetch_ohlcv, fetch_ticker_info
from src.nlp.text_sources import load_company_texts, load_category_texts

setup_logging(level="INFO")
logger = get_logger(__name__)

print("Project root      :", PROJECT_ROOT)
print("MongoDB URI       :", settings.MONGODB_URI)
print("Database          :", settings.MONGODB_DB_NAME)
print("DAILY_UPDATE_HOUR  :", settings.DAILY_UPDATE_HOUR)
print("Candidate tickers :", len(TICKERS))

Project root      : c:\Users\hp\Documents\GitHub\BeneathTheMarket
MongoDB URI       : mongodb://localhost:27017
Database          : tech_fintech_market_intel
DAILY_UPDATE_HOUR  : 18
Candidate tickers : 39


In [3]:
## 1. Local MongoDB connectivity test

def test_mongodb_connection() -> Dict[str, Any]:
    result = {
        "uri": settings.MONGODB_URI,
        "db_name": settings.MONGODB_DB_NAME,
        "reachable": False,
        "error": None,
        "server_info": None,
        "collections": [],
    }
    try:
        client = get_client()
        info = client.server_info()
        result["reachable"] = True
        result["server_info"] = {"version": info.get("version"), "ok": info.get("ok")}
        db = get_db()
        result["collections"] = db.list_collection_names()
        ensure_indexes(db)
        logger.info("MongoDB OK – version %s", info.get("version"))
    except Exception as exc:
        result["error"] = str(exc)
        logger.error("MongoDB connection failed: %s", exc)
    return result

mongo_status = test_mongodb_connection()
display(pd.Series(mongo_status).to_frame("value"))

if not mongo_status["reachable"]:
    raise RuntimeError(
        "Local MongoDB is not reachable. "
        "Start mongod on localhost:27017 before continuing."
    )

2026-10-12 00:57:12 | INFO     | src.db.connection | Connected to local MongoDB at mongodb://localhost:27017 (db=tech_fintech_market_intel)
2026-10-12 00:57:12 | INFO     | src.db.collections | All MongoDB indexes ensured.
2026-10-12 00:57:12 | INFO     | __main__ | MongoDB OK – version 8.2.3


,value
uri,mongodb://localhost:27017
db_name,tech_fintech_market_intel
reachable,True
error,None
server_info,"{'version': '8.2.3', 'ok': 1.0}"
collections,[]


In [4]:
## 2. Candidate universe overview

universe_rows = []
for cat, tickers in CATEGORIES.items():
    for t in tickers:
        universe_rows.append({"ticker": t, "primary_category": cat})

universe_df = pd.DataFrame(universe_rows)
print(f"Total candidate tickers: {len(universe_df)}")
print("\nTickers per category:")
display(universe_df.groupby("primary_category").size().rename("count").to_frame())
display(universe_df)

Total candidate tickers: 39

Tickers per category:


,count
primary_category,
Cloud and enterprise infrastructure,6
"Data, AI, and digital infrastructure software",6
Enterprise software and developer platforms,8
Fintech and digital financial infrastructure,6
Major technology platforms,6
Semiconductors and chip design,7


,ticker,primary_category
0,AAPL,Major technology platforms
1,MSFT,Major technology platforms
2,AMZN,Major technology platforms
3,GOOGL,Major technology platforms
4,META,Major technology platforms
5,NFLX,Major technology platforms
6,NVDA,Semiconductors and chip design
7,AMD,Semiconductors and chip design
8,AVGO,Semiconductors and chip design
9,QCOM,Semiconductors and chip design


In [5]:
## 3. yfinance availability & history check

def validate_ticker_yfinance(
    ticker: str,
    *,
    history_period: str = "5y",
    interval: str = "1d",
    recent_days: int = 10,
) -> Dict[str, Any]:
    ticker = ticker.upper()
    out: Dict[str, Any] = {
        "ticker": ticker,
        "category": get_primary_category(ticker),
        "success": False,
        "n_rows": 0,
        "first_ts": None,
        "last_ts": None,
        "columns": [],
        "error": None,
        "info_name": None,
        "info_currency": None,
        "info_exchange": None,
        "recent_rows": 0,
    }
    try:
        info = fetch_ticker_info(ticker)
        out["info_name"] = info.get("long_name") or info.get("short_name")
        out["info_currency"] = info.get("currency")
        out["info_exchange"] = info.get("exchange")

        df = fetch_ohlcv(ticker, period=history_period, interval=interval)
        if df is None or df.empty:
            out["error"] = "Empty DataFrame from yfinance"
            return out

        out["success"] = True
        out["n_rows"] = len(df)
        out["first_ts"] = str(df.index.min())
        out["last_ts"] = str(df.index.max())
        out["columns"] = list(df.columns)

        recent = fetch_ohlcv(
            ticker,
            start=datetime.utcnow() - timedelta(days=recent_days),
            interval=interval,
        )
        out["recent_rows"] = 0 if recent is None else len(recent)
    except Exception as exc:
        out["error"] = str(exc)
        logger.warning("Validation failed for %s: %s", ticker, exc)
    return out

In [6]:
results: List[Dict[str, Any]] = []
for i, ticker in enumerate(TICKERS, 1):
    print(f"[{i:02d}/{len(TICKERS)}] Validating {ticker} ...", end=" ", flush=True)
    res = validate_ticker_yfinance(ticker)
    results.append(res)
    status = "OK" if res["success"] else f"FAIL ({res['error']})"
    print(status)

validation_df = pd.DataFrame(results)
validation_df = validation_df.sort_values(["success", "category", "ticker"])
display(validation_df)

[01/39] Validating AAPL ... OK
[02/39] Validating MSFT ... 

C:\Users\hp\AppData\Local\Temp\ipykernel_24944\3685873217.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start=datetime.utcnow() - timedelta(days=recent_days),


OK
[03/39] Validating AMZN ... 

C:\Users\hp\AppData\Local\Temp\ipykernel_24944\3685873217.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start=datetime.utcnow() - timedelta(days=recent_days),


OK
[04/39] Validating GOOGL ... 

...

C:\Users\hp\AppData\Local\Temp\ipykernel_24944\3685873217.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start=datetime.utcnow() - timedelta(days=recent_days),


OK
[39/39] Validating CRWD ... 

C:\Users\hp\AppData\Local\Temp\ipykernel_24944\3685873217.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start=datetime.utcnow() - timedelta(days=recent_days),
C:\Users\hp\AppData\Local\Temp\ipykernel_24944\3685873217.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start=datetime.utcnow() - timedelta(days=recent_days),


OK


,ticker,category,success,n_rows,first_ts,last_ts,columns,error,info_name,info_currency,info_exchange,recent_rows
29,FI,Fintech and digital financial infrastructure,False,0,None,None,[],Empty DataFrame from yfinance,None,None,None,0
16,ANET,Cloud and enterprise infrastructure,True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,"Arista Networks, Inc.",USD,NYQ,7
15,CSCO,Cloud and enterprise infrastructure,True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,"Cisco Systems, Inc.",USD,NMS,7
17,DELL,Cloud and enterprise infrastructure,True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,Dell Technologies Inc.,USD,NYQ,7
18,HPE,Cloud and enterprise infrastructure,True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,Hewlett Packard Enterprise Company,USD,NYQ,7
14,IBM,Cloud and enterprise infrastructure,True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,International Business Machines Corporation,USD,NYQ,7
13,ORCL,Cloud and enterprise infrastructure,True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,Oracle Corporation,USD,NYQ,7
38,CRWD,"Data, AI, and digital infrastructure software",True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,"CrowdStrike Holdings, Inc.",USD,NMS,7
35,DDOG,"Data, AI, and digital infrastructure software",True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,"Datadog, Inc.",USD,NMS,7
37,MDB,"Data, AI, and digital infrastructure software",True,1255,2021-10-11 00:00:00,2026-10-09 00:00:00,"[Open, High, Low, Close, Volume, Dividends, St...",None,"MongoDB, Inc.",USD,NGM,7


In [7]:
## 4. Keep / drop decision

n_total = len(validation_df)
n_ok = int(validation_df["success"].sum())
n_fail = n_total - n_ok

print(f"Total tickers tested : {n_total}")
print(f"Successful           : {n_ok}")
print(f"Failed               : {n_fail}")
print()

MIN_ROWS = 500  # ≈ 2 years of daily bars

keep_mask = validation_df["success"] & (validation_df["n_rows"] >= MIN_ROWS)
keep_df = validation_df.loc[keep_mask].copy()
drop_df = validation_df.loc[~keep_mask].copy()

print(f"Tickers kept (≥ {MIN_ROWS} daily bars): {len(keep_df)}")
print(f"Tickers dropped                     : {len(drop_df)}")

if not drop_df.empty:
    print("\nDropped tickers:")
    display(drop_df[["ticker", "category", "n_rows", "error"]])

print("\nKept tickers by category:")
display(
    keep_df.groupby("category")
    .size()
    .rename("count")
    .to_frame()
    .sort_values("count", ascending=False)
)

Total tickers tested : 39
Successful           : 38
Failed               : 1

Tickers kept (≥ 500 daily bars): 38
Tickers dropped                     : 1

Dropped tickers:


,ticker,category,n_rows,error
29,FI,Fintech and digital financial infrastructure,0,Empty DataFrame from yfinance



Kept tickers by category:


,count
category,
Enterprise software and developer platforms,8
Semiconductors and chip design,7
"Data, AI, and digital infrastructure software",6
Cloud and enterprise infrastructure,6
Major technology platforms,6
Fintech and digital financial infrastructure,5


In [8]:
## 5. text source check

text_map = load_company_texts(tickers=keep_df["ticker"].tolist())
cat_texts = load_category_texts()

text_coverage = []
for t in keep_df["ticker"]:
    raw = text_map.get(t, "")
    text_coverage.append({
        "ticker": t,
        "has_text": bool(raw and raw.strip()),
        "text_length": len(raw) if raw else 0,
        "preview": (raw[:120] + "…") if raw and len(raw) > 120 else raw,
    })

text_df = pd.DataFrame(text_coverage)
n_with_text = int(text_df["has_text"].sum())

print(f"Tickers with non-empty description: {n_with_text} / {len(text_df)}")
print(f"Static category descriptions available: {len(cat_texts)}")
display(text_df)

2026-10-12 00:59:47 | INFO     | src.nlp.text_sources | Loaded non-news text for 38/38 tickers
Tickers with non-empty description: 38 / 38
Static category descriptions available: 6


,ticker,has_text,text_length,preview
0,ANET,True,173,"Providers of enterprise hardware, networking, ..."
1,CSCO,True,173,"Providers of enterprise hardware, networking, ..."
2,DELL,True,173,"Providers of enterprise hardware, networking, ..."
3,HPE,True,173,"Providers of enterprise hardware, networking, ..."
4,IBM,True,173,"Providers of enterprise hardware, networking, ..."
5,ORCL,True,173,"Providers of enterprise hardware, networking, ..."
6,CRWD,True,162,"Software firms focused on data platforms, anal..."
7,DDOG,True,162,"Software firms focused on data platforms, anal..."
8,MDB,True,162,"Software firms focused on data platforms, anal..."
9,NET,True,162,"Software firms focused on data platforms, anal..."


In [9]:
## 6. Persist artefacts

from src.utils import ensure_dir, save_json

processed_dir = ensure_dir(settings.PROCESSED_DATA_DIR)

# Keep-list
keep_path = processed_dir / "validated_tickers.csv"
keep_df[["ticker", "category", "n_rows", "first_ts", "last_ts", "info_name"]].to_csv(
    keep_path, index=False
)
print(f"Saved keep-list → {keep_path}")

# Summary
summary = {
    "run_at": utc_now().isoformat(),
    "mongodb_reachable": mongo_status["reachable"],
    "mongodb_version": (mongo_status.get("server_info") or {}).get("version"),
    "candidate_tickers": n_total,
    "successful_fetches": n_ok,
    "kept_tickers": len(keep_df),
    "dropped_tickers": drop_df["ticker"].tolist(),
    "min_rows_threshold": MIN_ROWS,
    "text_coverage": n_with_text,
    "categories_kept": keep_df.groupby("category").size().to_dict(),
}

summary_path = processed_dir / "feasibility_summary.json"
save_json(summary, summary_path)
print(f"Saved feasibility summary → {summary_path}")
print("\n=== FEASIBILITY SUMMARY ===")
for k, v in summary.items():
    if k not in ("dropped_tickers", "categories_kept"):
        print(f"  {k}: {v}")

Saved keep-list → C:\Users\hp\Documents\GitHub\BeneathTheMarket\data\processed\validated_tickers.csv
Saved feasibility summary → C:\Users\hp\Documents\GitHub\BeneathTheMarket\data\processed\feasibility_summary.json

=== FEASIBILITY SUMMARY ===
  run_at: 2026-10-11T19:30:20.366188+00:00
  mongodb_reachable: True
  mongodb_version: 8.2.3
  candidate_tickers: 39
  successful_fetches: 38
  kept_tickers: 38
  min_rows_threshold: 500
  text_coverage: 38


## 7. Acceptance criteria

| Criterion | Status |
|-----------|--------|
| Local MongoDB reachable | Cell 3 |
| ≥ 30 tickers with usable history | Cell 10 |
| Non-news text source verified | Cell 12 |
| Keep-list & summary persisted | Cell 14 |


In [10]:
print("Notebook 01 finished successfully.")

Notebook 01 finished successfully.
